In [0]:
file_path = "/Volumes/workspace/default/chicagofoodinspection/export.csv"

df = spark.read.csv(
    file_path,
    header=True,
    inferSchema=False,
    multiLine=True,
    escape='"'
)

print("Total Rows:", df.count())
print("Total Columns:", len(df.columns))

display(df.limit(5))

In [0]:
df.printSchema()

In [0]:
from pyspark.sql import functions as F

missing_counts = df.select([
    F.sum(
        F.when(
            F.col(c).isNull() |
            (F.trim(F.col(c)) == ""),
            1
        ).otherwise(0)
    ).alias(c)
    for c in df.columns
])

display(missing_counts)

In [0]:
violation_analysis = (
    df.groupBy("Results")
    .agg(
        F.count("*").alias("total_inspections"),

        F.sum(
            F.when(
                F.col("Violations").isNull() |
                (F.trim(F.col("Violations")) == ""),
                1
            ).otherwise(0)
        ).alias("missing_violations")
    )
    .withColumn(
        "missing_percentage",
        F.round(
            F.col("missing_violations") * 100 /
            F.col("total_inspections"),
            2
        )
    )
    .orderBy(F.desc("total_inspections"))
)

display(violation_analysis)

In [0]:
df.dropna()

In [0]:
total_rows = df.count()

unique_ids = df.select("Inspection ID").distinct().count()

duplicate_ids = total_rows - unique_ids
print("Total Records:", total_rows)
print("Unique Inspection IDs:", unique_ids)
print("Duplicate Inspection IDs:", duplicate_ids)

In [0]:
df.select(
    "Inspection ID",
    "Inspection Date",
    "Latitude",
    "Longitude",
    "License #",
    "Zip"
).show(10, truncate=False)

In [0]:
from pyspark.sql import functions as F

silver_df =(
    df
    .withColumn(
        "Inspection ID",
        F.expr("try_cast('INSPECTION ID' AS BIGINT)")
    )
    .withColumn(
        "Inspection_Date",
        F.try_to_timestamp(
            F.col("Inspection Date"),
            F.lit("MM/dd/yyyy")
        ).cast("date")
    )
    .withColumn(
        "Latitude",
        F.expr("try_cast('Latitude' AS DOUBLE)")
    )
    .withColumn(
        "Longitude",
        F.expr("try_cast('Longitude' AS DOUBLE)")
    )
)

silver_df.printSchema()

In [0]:
silver_df = (
    silver_df
    .drop("Inspection Date")
    .withColumnRenamed("Inspection_Date", "Inspection Date")
)

silver_df.select("Inspection Date").show(5)

silver_df.printSchema()

In [0]:
from pyspark.sql import functions as F

date_test = df.select(
    F.col("Inspection Date").alias("Original_Date"),
    F.to_date(
        F.col("Inspection Date"),
        "MM/dd/yyyy"
    ).alias("Converted_Date")
)

date_test.show(5)

date_test.printSchema()

In [0]:
silver_df = silver_df.withColumn(
    "Inspection Date",
    F.to_date(
        F.col("Inspection Date"),
        "MM/dd/yyyy"
    )
)

silver_df.select("Inspection Date").show(5)

print("Data type:", silver_df.schema["Inspection Date"].dataType)
print("Total columns:", len(silver_df.columns))

In [0]:
columns_to_check = [
    "Inspection ID",
    "Inspection Date",
    "Latitude",
    "Longitude"
]

raw_missing = df.select([
    F.sum(
        F.when(
            F.col(c).isNull() |
            (F.trim(F.col(c)) == ""),
            1
        ).otherwise(0)
    ).alias(c)
    for c in columns_to_check
]).first().asDict()

silver_missing = silver_df.select([
    F.sum(
        F.col(c).isNull().cast("int")
    ).alias(c)
    for c in columns_to_check
]).first().asDict()

for c in columns_to_check:
    print(
        c,
        "| Raw Missing:", raw_missing[c],
        "| Silver Missing:", silver_missing[c],
        "| Addtional Nulls:",
        silver_missing[c] - raw_missing[c]
    )


In [0]:
from pyspark.sql import functions as F

conversion_test = df.select(
    F.col("Inspection ID").alias("Original_ID"),

    F.expr(
        "try_cast('Inspection ID' AS BIGINT)"
    ).alias("Converted_ID"),

    F.col("Latitude").alias("Original_Latitude"),

    F.expr(
        "try_cast('Latitude' AS DOUBLE)"
    ).alias("Converted_Latitude"),

    F.col("Longitude").alias("Original_Longitude"),

    F.expr(
        "try_cast('Longitude' AS DOUBLE)"
    ).alias("Converted_Longitude")
)

conversion_test.show(5, truncate=False)

In [0]:
from pyspark.sql import functions as F

test_df = df.select(
    F.col("Inspection ID").cast("long").alias("Inspection_ID"),
    F.col("Latitude").cast("double").alias("Latitude"),
    F.col("Longitude").cast("double").alias("Longitude")
)

test_df.limit(5).show(truncate=False)

In [0]:
from pyspark.sql import functions as F

silver_df = (
    df
    .withColumn(
        "Inspection ID",
        F.col("Inspection ID").cast("long")
    )
    .withColumn(
        "Inspection Date",
        F.to_date(F.col("Inspection Date"), "MM/dd/yyyy")
    )
    .withColumn(
        "Latitude",
        F.col("Latitude").cast("double")
    )
    .withColumn(
        "Longitude",
        F.col("Longitude").cast("double")
    )
)

silver_df.printSchema()

silver_df.select(
    "Inspection ID",
    "Inspection Date",
    "Latitude",
    "Longitude"
).show(5)

In [0]:
from pyspark.sql import functions as F

columns_to_check = [
    "Inspection ID",
    "Inspection Date",
    "Latitude",
    "Longitude"
]

validation = silver_df.select([
    F.sum(
        F.when(F.col(c).isNull(), 1)
        .otherwise(0)
    ).alias(c)
    for c in columns_to_check
])

display(validation)


In [0]:
# Check Risk categories
silver_df.groupBy("Risk") \
    .count() \
    .orderBy(F.desc("count")) \
    .show(30, truncate=False)

# Check Results categories
silver_df.groupBy("Results") \
    .count() \
    .orderBy(F.desc("count")) \
    .show(30, truncate=False)

In [0]:
# Check city names
silver_df.groupBy("City") \
    .count() \
    .orderBy(F.desc("count")) \
    .show(30, truncate=False)

# Check state values
silver_df.groupBy("State") \
    .count() \
    .orderBy(F.desc("count")) \
    .show(20, truncate=False)

In [0]:
from pyspark.sql import functions as F

silver_df = (
    silver_df
    .withColumn(
        "City",
        F.upper(F.trim(F.col("City")))
    )
    .withColumn(
        "State",
        F.upper(F.trim(F.col("State")))
    )
)

silver_df.groupBy("City") \
    .count() \
    .orderBy(F.desc("count")) \
    .show(20, truncate=False)

In [0]:
suspected_cities = [
    "CCHICAGO",
    "CHICAGOCHICAGO",
    "CHICAGOO",
    "CHICAGO.",
    "312CHICAGO",
    "CHCHICAGO",
    "INACTIVE",
    "MERRIVILLE",
    "NILES NILES"
]

city_analysis = (
    silver_df
    .filter(F.col("City").isin(suspected_cities))
    .groupBy("City", "State")
    .agg(
        F.count("*").alias("records"),
        F.sort_array(
            F.collect_set("Zip")
        ).alias("zip_codes")
    )
    .orderBy(F.desc("records"))
)

display(city_analysis)


In [0]:
from pyspark.sql import functions as F

city_corrections = {
    "CCHICAGO": "CHICAGO",
    "CHICAGOCHICAGO": "CHICAGO",
    "CHICAGOO": "CHICAGO",
    "CHICAGO.": "CHICAGO",
    "312CHICAGO": "CHICAGO",
    "CHCHICAGO": "CHICAGO",
    "MERRIVILLE": "MERRILLVILLE",
    "NILES NILES": "NILES",
}

mapping = F.create_map(
    *[
        item
        for old, new in city_corrections.items()
        for item in (F.lit(old), F.lit(new))
    ]
)

silver_df = (
    silver_df
    .withColumn("City_Raw", F.col("City"))
    .withColumn(
        "City",
        F.coalesce(
            F.element_at(mapping, F.col("City")),
            F.col("City")
        )
    )
)

In [0]:
silver_df.filter(
    F.col("City").isin(
        "CHICAGO",
        "MERRILLVILLE",
        "NILES",
        "INACTIVE"
    )
).groupBy("City").count().orderBy(
    F.desc("count")
).show()

In [0]:
silver_df = silver_df.withColumn(
    "City_Quality_Flag",
    F.when(
        F.col("City").isNull() |
        (F.trim(F.col("City")) == ""),
        "MISSING_CITY"
    )
    .when(
        F.col("City") == "INACTIVE",
        "INVALID_CITY"
    )
    .otherwise("NOT_FLAGGED")
)

silver_df.groupBy("City_Quality_Flag") \
    .count() \
    .show()

In [0]:
state_zip_analysis = (
    silver_df
    .groupBy("State", "City", "Zip")
    .count()
    .filter(
        (F.col("State") != "IL") |
        F.col("State").isNull()
    )
    .orderBy(F.desc("count"))
)

display(state_zip_analysis)

In [0]:
from pyspark.sql import functions as F

silver_df = silver_df.withColumn(
    "State_Quality_Flag",

    F.when(
        F.col("State").isNull() |
        (F.trim(F.col("State")) == ""),
        "MISSING_STATE"
    )

    .when(
        (F.col("Zip") == "60642") &
        (F.col("State") != "IL"),
        "STATE_ZIP_MISMATCH"
    )

    .when(
        F.col("State") != "IL",
        "OUT_OF_STATE_REVIEW"
    )

    .otherwise("NOT_FLAGGED")
)

silver_df.groupBy("State_Quality_Flag") \
    .count() \
    .orderBy(F.desc("count")) \
    .show()

In [0]:
from pyspark.sql import functions as F

silver_df = silver_df.withColumn(
    "Zip_Quality_Flag",

    F.when(
        F.col("Zip").isNull() |
        (F.trim(F.col("Zip")) == ""),
        "MISSING_ZIP"
    )

    .when(
        ~F.trim(F.col("Zip")).rlike(
            r"^\d{5}(-\d{4})?$"
        ),
        "INVALID_ZIP_FORMAT"
    )

    .otherwise("NOT_FLAGGED")
)

silver_df.groupBy("Zip_Quality_Flag") \
    .count() \
    .orderBy(F.desc("count")) \
    .show()

In [0]:
from pyspark.sql import functions as F

silver_df.select(
    "Zip",
    F.trim(F.col("Zip")).alias("Clean_Zip"),
    F.length(F.trim(F.col("Zip"))).alias("length"),
    F.trim(F.col("Zip")).rlike(
        r"^[0-9]{5}(-[0-9]{4})?$"
    ).alias("Is_Valid")
).show(10, truncate=False)

In [0]:
silver_df = silver_df.withColumn(
    "Zip_Quality_Flag",

    F.when(
        F.col("Zip").isNull() |
        (F.trim(F.col("Zip")) == ""),
        "MISSING_ZIP"
    )

    .when(
        ~F.trim(F.col("Zip")).rlike(
            r"^[0-9]{5}(-[0-9]{4}})?$"
        ),
        "INVALID_ZIP_FORMAAT"
    )

    .otherwise("NOT_FLAGGED")
)

silver_df.groupBy("Zip_Quality_Flag") \
    .count() \
    .orderBy(F.desc("count")) \
    .show()

In [0]:
from pyspark.sql import functions as F

silver_df = silver_df.withColumn(
    "Coordinate_Quality_Flag",
    
    F.when(
        F.col("Latitude").isNull() &
        F.col("Longitude").isNull(),
        "MISSING_COORDINATES"
    )

    .when(
        F.col("Latitude").isNull() |
        F.col("Longitude").isNull(),
        "PARTIAL_COORDINATES"
    )

    .when(
        ~F.col("Latitude").between(-90, 90) |
        ~F.col("Longitude").between(-180,180),
        "INVALID_COORDINATES"
    )

    .otherwise("NOT_FLAGGED")
)

silver_df.groupBy("Coordinate_Quality_Flag") \
    .count() \
        .orderBy(F.desc("count")) \
        .show()

In [0]:
from pyspark.sql import functions as F

silver_df = silver_df.withColumn(
    "Overall_Quality_Status",

    F.when(
        (F.col("City_Quality_Flag") != "NOT_FLAGGED") |
        (F.col("State_Quality_Flag") != "NOT_FLAGGED") |
        (F.col("Zip_Quality_Flag") != "NOT_FLAGGED") |
        (F.col("Coordinate_Quality_Flag").isin(
            "INVALID_COORDINATES",
            "PARTIAL_COORDINATES"
        )),
        "REVIEW_REQUIRED"
    )

    .when(
        F.col("Coordinate_Quality_Flag") ==
        "MISSING_COORDINATES",
        "USABLE_WITHOUT_GEO"
    )

    .otherwise("NOT_FLAGGED")
)

silver_df.groupBy("Overall_Quality_Status") \
    .count() \
    .orderBy(F.desc("count")) \
    .show()

In [0]:
from pyspark.sql import functions as F

print("Raw Records:", silver_df.count())

print(
    "Unique Inspection IDs:",
    silver_df.select("Inspection ID").distinct().count()
)

print(
    "Missing Inspection IDs:",
    silver_df.filter(
        F.col("Inspection ID").isNull()
    ).count()
)

print(
    "Missing Inspection Dates:",
    silver_df.filter(
        F.col("Inspection Date").isNull()
    ).count()
)

In [0]:
print("Raw Records:", df.count())
print("Silver Records:", silver_df.count())

assert df.count() == silver_df.count(), \
    "Data loss detected during cleaning!"

print("Row count validation passed!")

In [0]:
import re

# Rename columns for Delta compatibility
bronze_columns = [
    re.sub(r"[^a-zA-Z0-9]+", "_", c)
      .strip("_")
      .lower()
    for c in df.columns
]

# Preserve original raw values
bronze_df = df.toDF(*bronze_columns)

# Save as a Delta table
bronze_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "workspace.default.chicago_food_inspections_bronze"
    )

print("Bronze Delta table created successfully!")

In [0]:
bronze_table = spark.table(
    "workspace.default.chicago_food_inspections_bronze"
)

print("Bronze Records:", bronze_table.count())
print("Bronze Columns:", len(bronze_table.columns))

In [0]:
import re
silver_columns = [
    re.sub(r"[^a-zA-Z0-9]+", "_", c)
        .strip("_")
        .lower()
    for c in silver_df.columns
]

silver_delta_df = silver_df.toDF(*silver_columns)

silver_delta_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(
        "workspace.default.chicago_food_inpsections_silver"
    )
print("Silver Delta table created successfullt!")

In [0]:
spark.sql(
    "SHOW TABLES IN workspace.default"
).show(truncate=False)

In [0]:
spark.sql("""
ALTER TABLE workspace.default.chicago_food_inpsections_silver RENAME TO workspace.default.chicago_food_inspection_silver
""")

In [0]:
silver_table = spark.table(
    "workspace.default.chicago_food_inspection_silver"
)

print("Silver Records:", silver_table.count())
print("Silver Columns:", len(silver_table.columns))

silver_table.groupBy(
    "overall_quality_status"
).count().show()

In [0]:
from pyspark.sql import functions as F

gold_risk_df = (
    silver_table

    # Only completed inspections
    .filter(
        F.col("results").isin(
            "Pass", "Fail", "Pass w/ Conditions"
        )
    )

    # Only defined risk levels
    .filter(
        F.col("risk").isin(
            "Risk 1 (High)",
            "Risk 2 (Medium)",
            "Risk 3 (Low)"
        )
    )

    # Aggregate by risk
    .groupBy("risk")
    .agg(
        F.count("*").alias("total_inspections"),

        F.sum(
            F.when(F.col("results") == "Fail", 1)
            .otherwise(0)
        ).alias("failed_inspections")
    )

    # Calculate failure percentage
    .withColumn(
        "failure_rate_pct",
        F.round(
            F.col("failed_inspections") * 100 /
            F.col("total_inspections"),
            2
        )
    )
)

display(gold_risk_df)

In [0]:
gold_risk_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(
        "workspace.default.chicago_food_inspection_gold_risk"
    )
print("Gold risk analytics table created")

In [0]:
gold_table = spark.table(
    "workspace.default.chicago_food_inspection_gold_risk"
)

display(gold_table)

In [0]:
from pyspark.sql import functions as F

gold_yearly_df = (
    silver_table
    .filter(
        F.col("results").isin(
            "Pass",
            "Fail",
            "Pass w/ Conditions"
        )
    )
    .withColumn(
        "inspection_year",
        F.year(F.col("inspection_date"))
    )
    .groupBy("inspection_year")
    .agg(
        F.count("*").alias("total_inspections"),
        F.sum(
            F.when(F.col("results") == "Fail", 1)
            .otherwise(0)
        ).alias("failed_inspections")
    )
    .withColumn(
        "failure_rate_pct",
        F.round(
            F.col("failed_inspections") * 100 /
            F.col("total_inspections"),
            2
        )
    )
    .orderBy("inspection_year")
)

display(gold_yearly_df)

In [0]:
gold_yearly_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(
        "workspace.default.chicago_inspection_yearly_gold"
    )

print("Yearly Gold Delta table created!")

In [0]:
yearly_table = spark.table(
    "workspace.default.chicago_inspection_yearly_gold"
)

print("Total Years:", yearly_table.count())

print(
    "Total Completed Inspections:",
    yearly_table.agg(
        F.sum("total_inspections")
    ).first()[0]
)

display(yearly_table.orderBy("inspection_year"))

In [0]:
# Validate Gold totals against Silver records

silver_completed = silver_table.filter(
    F.col("results").isin(
        "Pass", "Fail", "Pass w/ Conditions"
    )
)

silver_total = silver_completed.count()

silver_failed = silver_completed.filter(
    F.col("results") == "Fail"
).count()

gold_totals = yearly_table.agg(
    F.sum("total_inspections").alias("total"),
    F.sum("failed_inspections").alias("failed")
).first()

print("Silver Total:", silver_total)
print("Gold Total:", gold_totals["total"])

print("Silver Failed:", silver_failed)
print("Gold Failed:", gold_totals["failed"])

assert silver_total == gold_totals["total"]
assert silver_failed == gold_totals["failed"]

print("Gold Layer Validation Passed!")

In [0]:
silver_details = spark.sql("""
    DESCRIBE DETAIL
    workspace.default.chicago_food_inspection_silver
""")

silver_details.select(
    "format",
    "numFiles",
    "sizeInBytes",
    "partitionColumns"
).show(truncate=False)